# WhatsApp order parser | Colab

Explore episode boundaries, then compare a word-TF-IDF baseline, a tuned classical pipeline, and joint MuRIL on the same held-out phones. These predictions use complete episodes and measure agreement with reviewed silver labels. Status is scored both on known order-related episodes and after the relevance filter.

1. Select a TPU runtime. This notebook uses one XLA device and reports the detected hardware. The attached run used v5e-1. An eight-chip slice is v5e-8; this notebook uses one device on a larger single-host slice.
2. Put **colab-episodes.jsonl.gz** next to the project folders on Drive and set **PROJECT** below. The notebook reads the archive in one sequential copy and uses local disk afterward.
3. Run all. If updating dependencies, restart the runtime once before proceeding. An existing compatible checkpoint under **models/muril-joint-v1** skips fine-tuning. Keep Drive private because the archive contains pharmacy conversations.

Validation selects the classical settings, MuRIL epoch, and each relevance cutoff. Test phones are used only for the final comparison. The 99% order-recall floor applies to validation.

In [ ]:
# Install only if the runtime does not already have matching Torch and XLA.
import os
# This notebook imports torch_xla explicitly. Skip Colab's separate torch_tpu entry point.
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import importlib.metadata as metadata
import subprocess
import sys

def version(name):
    try:
        return metadata.version(name).split('+')[0]
    except metadata.PackageNotFoundError:
        return None

if version('torch') is None or version('torch') != version('torch-xla'):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'torch==2.8.0', 'torch_xla[tpu]==2.8.0',
                           'torchvision==0.23.0', 'torchaudio==2.8.0',
                           '-f', 'https://storage.googleapis.com/libtpu-releases/index.html'])
if version('transformers') != '4.56.2':
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.56.2'])

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/whatsapp-order-parser')
RUN = PROJECT / 'models/muril-joint-v1'

import gzip
import hashlib
import json
import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (average_precision_score, f1_score, precision_recall_curve,
                             precision_recall_fscore_support, confusion_matrix)
BASE = 'google/muril-base-cased'
STATUSES = ['inquiry', 'confirmed', 'out_for_delivery', 'completed', 'cancelled']
SEED, LENGTH, BATCH, EPOCHS = 42, 384, 8, 4
RECALL_TARGET = 0.99

archive = PROJECT / 'colab-episodes.jsonl.gz'
local_archive = Path('/content/colab-episodes.jsonl.gz')
shutil.copyfile(archive, local_archive)
rows = []
digest = hashlib.sha256()
with gzip.open(local_archive, 'rt', encoding='utf-8') as source:
    timelines = json.loads(next(source))['timelines']
    for line in source:
        e = json.loads(line)
        target = STATUSES.index(e['order_status']) if e['order_status'] is not None else -100
        text = e['text']
        digest.update(json.dumps([e['episode_id'], e['phone'], target, text]).encode())
        rows.append((text, e['phone'], target))

RUN.mkdir(parents=True, exist_ok=True)
groups = np.array([r[1] for r in rows])
status = np.array([r[2] for r in rows])
order = (status >= 0).astype(int)
def split(ids, folds):
    a, b = next(StratifiedGroupKFold(folds, shuffle=True, random_state=SEED).split(ids, status[ids], groups[ids]))
    return ids[a], ids[b]
dev, test = split(np.arange(len(rows)), 5)
train, valid = split(dev, 4)
for a, b in [(train, valid), (train, test), (valid, test)]:
    assert set(groups[a]).isdisjoint(groups[b])
for ids in [train, valid, test]:
    assert set(status[ids]) == {-100, *range(5)}
print(pd.DataFrame({k: pd.Series(status[ix]).value_counts() for k, ix in
                    [('train', train), ('validation', valid), ('test', test)]}).rename(index={-100:'unrelated', **dict(enumerate(STATUSES))}))
signature = [digest.hexdigest(), BASE, LENGTH, SEED, BATCH, EPOCHS, STATUSES,
             hashlib.sha256(np.concatenate([train, [-1], valid, [-1], test]).tobytes()).hexdigest()]
metadata = RUN / 'run.json'
if metadata.exists() and json.loads(metadata.read_text()) != signature:
    raise ValueError('Dataset or training configuration changed. Choose a new RUN directory.')
metadata.write_text(json.dumps(signature))

## How messages become episodes

The same customer can ask about stock, confirm an order, and share delivery details hours apart. Splitting at every pause would fragment those exchanges. A customer may also come back days or weeks later with a new request.

For each phone, sort its messages by timestamp and calculate the mean gap between consecutive messages. Start a new episode when a gap is **strictly greater than the larger of 36 hours and that phone's mean gap**. The 36-hour minimum helps keep overnight pauses together. The phone-specific mean allows more time between messages in chats that are naturally spaced out. A chat with one message forms one episode.

The timelines compare the mean-only rule with the 36-hour minimum. Each dot is a message and colors alternate at episode boundaries. Examples are chosen to show where the floor changes a short chat and how longer chats behave. This is a time-based heuristic. The modeling sections use the frozen, reviewed episodes.

In [ ]:
def gap_clusters(times, floor):
    gaps = (times[1:] - times[:-1]).total_seconds().to_numpy() / 3600
    cutoff = max(floor, gaps.mean()) if gaps.size else floor
    return np.r_[0, np.cumsum(gaps > cutoff)]

examples = []
for values in timelines:
    times = pd.to_datetime(values, utc=True, format='mixed').sort_values()
    mean_only = gap_clusters(times, 0)
    with_floor = gap_clusters(times, 36)
    examples.append((times, mean_only, with_floor))
changed = [row for row in examples if row[1][-1] > row[2][-1]]
recurring = sorted((row for row in examples if row[2][-1] >= 1),
                   key=lambda row: (row[2][-1], len(row[0])))
focus = [min(changed, key=lambda row: len(row[0])), recurring[len(recurring)//2], recurring[-1]]

colors = ['#246b9b', '#bf5b20']
fig, axes = plt.subplots(3, 2, figsize=(13, 6), layout='constrained')
summary = []
for i, (times, before, after) in enumerate(focus):
    summary.append([f'Chat {i+1}', len(times), before[-1]+1, after[-1]+1])
    start = (times - times[0]).total_seconds() / 3600
    xmax = max(start[-1], 1)
    for j, clusters in enumerate([before, after]):
        ax = axes[i, j]
        ax.scatter(start, np.zeros(len(times)), c=[colors[k % 2] for k in clusters], s=22)
        ax.set(xlim=(-xmax*.03, xmax*1.03), yticks=[], ylabel=f'Chat {i+1}' if j == 0 else '',
               xlabel='Hours since first message' if i == 2 else '')
        for boundary in np.flatnonzero(np.diff(clusters)) + 1:
            ax.axvline((start[boundary-1] + start[boundary])/2, color='gray', alpha=.35, lw=1)
axes[0,0].set_title('Mean gap only')
axes[0,1].set_title('Mean gap with 36-hour minimum')
plt.show()
display(pd.DataFrame(summary, columns=['Example', 'Messages', 'Episodes with mean only',
                                       'Episodes with 36-hour minimum']))

In [ ]:
def select_order_cutoff(truth, order_scores, recall_floor):
    def key(t):
        keep = order_scores >= t
        recall = keep[truth == 1].mean()
        filtered = (~keep[truth == 0]).mean()
        return recall >= recall_floor, filtered, recall
    return float(max(np.unique(order_scores), key=key))

from itertools import product
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from threadpoolctl import threadpool_limits

def fit_comparator(tuned):
    word = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=3, sublinear_tf=True)
    vec = FeatureUnion([('word', word), ('char', char)]) if tuned else word
    x = vec.fit_transform([rows[i][0] for i in train])
    v = vec.transform([rows[i][0] for i in valid])
    z = vec.transform([rows[i][0] for i in test])
    grid = list(product((0.1, 1, 10), (None, 'balanced'))) if tuned else [(1, None)]
    best_ap = best_f1 = -1
    train_orders = order[train] == 1
    valid_orders = order[valid] == 1
    for c, weight in grid:
        args = dict(C=c, class_weight=weight, random_state=SEED)
        gate = LinearSVC(**args, max_iter=20000) if tuned else LogisticRegression(**args, max_iter=500)
        gate.fit(x, order[train])
        v_score = gate.decision_function(v)
        ap = average_precision_score(1-order[valid], -v_score)
        if ap > best_ap:
            best_ap, best_gate, best_v = ap, gate, v_score
        stage = LogisticRegression(**args, max_iter=500).fit(x[train_orders], status[train][train_orders])
        mf = f1_score(status[valid][valid_orders], stage.predict(v[valid_orders]), average='macro')
        if mf > best_f1:
            best_f1, best_stage = mf, stage
    cutoff = select_order_cutoff(order[valid], best_v, RECALL_TARGET)
    print('Tuned' if tuned else 'Baseline', '| relevance:', best_gate, '| status:', best_stage)
    return best_gate.decision_function(z), cutoff, best_stage.predict(z)

with threadpool_limits(limits=1, user_api='blas'):
    systems = {'Word baseline': fit_comparator(False), 'Tuned classical': fit_comparator(True)}

## Joint MuRIL

One encoder serves both tasks. The status loss applies only to order-related episodes. The attached run's status macro F1 reached about 0.50 by epoch 3, while unrelated average precision peaked in epoch 1. That gives little evidence for a hyperparameter change. This rewrite keeps the existing settings and resumes a compatible unfinished checkpoint or loads the completed one.

In [ ]:
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'
os.environ['PJRT_DEVICE'] = 'TPU'
import torch
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.runtime as xr
from torch import nn
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer, AutoConfig, set_seed, get_linear_schedule_with_warmup

device = torch_xla.device()
if xr.device_type() != 'TPU':
    raise RuntimeError('Select a TPU runtime in Colab before running this cell.')
print('Using', xm.xla_device_hw(device), device)

def atomic_save(obj, path):
    tmp = path.with_suffix('.tmp')
    xm.save(obj, str(tmp))
    os.replace(tmp, path)

set_seed(SEED)
tokenizer_dir = RUN / 'tokenizer'
tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir if (tokenizer_dir/'tokenizer.json').exists() else BASE)
if not (tokenizer_dir/'tokenizer.json').exists():
    tokenizer.save_pretrained(tokenizer_dir)

def encode(text):
    ids = tokenizer.encode(text, add_special_tokens=False, truncation=False, verbose=False)
    budget = LENGTH - tokenizer.num_special_tokens_to_add(pair=False)
    # Keep the opening request and the latest fulfillment evidence.
    if len(ids) > budget:
        ids = ids[:budget//2] + ids[-(budget-budget//2):]
    return {'input_ids': tokenizer.build_inputs_with_special_tokens(ids)}

encoded = [encode(r[0]) for r in rows]
def collate(ix):
    return tokenizer.pad([encoded[i] for i in ix], padding='max_length', max_length=LENGTH, return_tensors='pt'), torch.tensor(order[ix]), torch.tensor(status[ix])
def loader(ix, epoch=None):
    return DataLoader(ix.tolist(), batch_size=BATCH, collate_fn=collate, shuffle=epoch is not None,
                      generator=torch.Generator().manual_seed(SEED + (epoch or 0)))
class JointMuRIL(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.dropout = nn.Dropout(0.1)
        self.binary = nn.Linear(encoder.config.hidden_size, 2)
        self.status = nn.Linear(encoder.config.hidden_size, len(STATUSES))
    def forward(self, batch):
        pooled = self.dropout(self.encoder(**batch).last_hidden_state[:, 0])
        return self.binary(pooled), self.status(pooled)

best_file, resume_file, done = [RUN / name for name in ('best.pt','resume.pt','finished.json')]
if done.exists() or resume_file.exists():
    encoder = AutoModel.from_config(AutoConfig.from_pretrained(RUN/'config'), attn_implementation='eager')
else:
    encoder = AutoModel.from_pretrained(BASE, attn_implementation='eager')
    encoder.config.save_pretrained(RUN/'config')
model = JointMuRIL(encoder).to(device)
def move(batch):
    return {k:v.to(device) for k,v in batch.items()}

# Replaced torch.inference_mode() with torch.no_grad() to avoid version counter errors on TPU
@torch.no_grad()
def scores(ix):
    model.eval()
    binary, stages = [], []
    for x, _, _ in loader(ix):
        b, s = model(move(x))
        binary.extend(b.softmax(-1)[:,1].cpu().tolist())
        stages.extend(s.softmax(-1).cpu().tolist())
    return np.array(binary), np.array(stages)

In [ ]:
if not done.exists():
    opt = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    steps = len(loader(train)) * EPOCHS
    schedule = get_linear_schedule_with_warmup(opt, int(.1 * steps), steps)
    # Square-root inverse frequency tempers the influence of the smallest classes.
    def criterion(y):
        w = np.bincount(y) ** -0.5
        return nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=device))
    binary_loss = criterion(order[train])
    stage_loss = criterion(status[train][order[train] == 1])


    start, best = 0, -1.
    if resume_file.exists():
        ck = torch.load(resume_file, map_location='cpu', weights_only=True)
        model.load_state_dict(ck['model'])
        opt.load_state_dict(ck['optimizer'])
        schedule.load_state_dict(ck['schedule'])
        start, best = ck['epoch'], ck['best']
        del ck
    for epoch in range(start, EPOCHS):
        set_seed(SEED + epoch)
        model.train()
        for x, y, s in loader(train, epoch):
            y, s = y.to(device), s.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('xla', dtype=torch.bfloat16):
                b, logits = model(move(x))
                loss = binary_loss(b, y)
                # Keep a fixed tensor shape, including batches with no status labels.
                per_row = nn.functional.cross_entropy(logits.float(), s, weight=stage_loss.weight,
                                                      ignore_index=-100, reduction='none')
                valid_weight = stage_loss.weight[s.clamp_min(0)] * (s >= 0)
                loss = loss + per_row.sum() / valid_weight.sum().clamp_min(1e-8)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.)
            opt.step()
            schedule.step()
            torch_xla.sync()
        p, q = scores(valid)
        mask = order[valid] == 1
        ap = average_precision_score(1-order[valid], 1-p)
        mf = f1_score(status[valid][mask], q[mask].argmax(1), average='macro', zero_division=0)
        metric = float((ap + mf)/2)
        if metric > best:
            best = metric
            atomic_save(model.state_dict(), best_file)
        atomic_save(dict(epoch=epoch+1, best=best, model=model.state_dict(), optimizer=opt.state_dict(),
                         schedule=schedule.state_dict()), resume_file)
        print(f'Epoch {epoch+1}: validation unrelated AP {ap:.3f}, status macro F1 {mf:.3f}', flush=True)
    assert best_file.exists()
    tmp = done.with_suffix('.tmp')
    tmp.write_text(json.dumps({'best_validation_composite':best}))
    os.replace(tmp, done)
    resume_file.unlink(missing_ok=True)
model.load_state_dict(torch.load(best_file, map_location='cpu', weights_only=True))
model.eval()
print('Loaded selected checkpoint:', best_file)

## Final comparison

The first chart shows how well each system ranks unrelated episodes. The second shows the operational tradeoff at each validation-selected threshold. The status table and per-class chart expose performance on rare statuses. Route-everything is the business reference point at zero orders missed and zero unrelated episodes filtered.

In [ ]:
validation_order, _ = scores(valid)
threshold = select_order_cutoff(order[valid], validation_order, RECALL_TARGET)
p, q = scores(test)
systems['MuRIL'] = (p, threshold, q.argmax(axis=1))
decision = RUN / 'decision.json'
decision.write_text(json.dumps(dict(threshold=threshold, target=RECALL_TARGET, statuses=STATUSES)))
from sklearn.metrics import precision_recall_fscore_support

truth = order[test]
related = truth == 1
binary_rows, status_rows, stage_f1 = {}, {}, {}
fig, (ax, tradeoff) = plt.subplots(1, 2, figsize=(13, 4), layout='constrained')
for name, (score, cutoff, stage) in systems.items():
    keep = (score >= cutoff).astype(int)
    precision, recall, _, _ = precision_recall_fscore_support(truth, keep, labels=[0, 1], zero_division=0)
    binary_rows[name] = {'Unrelated AP': average_precision_score(1-truth, -score),
                         'Order precision': precision[1], 'Order recall': recall[1],
                         'Unrelated precision': precision[0], 'Unrelated recall': recall[0],
                         'Orders missed': int(sum(related & (keep == 0))),
                         'Unrelated filtered': int(sum(~related & (keep == 0)))}
    stage_f1[name] = f1_score(status[test][related], stage[related], labels=range(5),
                              average=None, zero_division=0)
    status_rows[name] = {'Conditional status macro F1': stage_f1[name].mean(),
                         'End-to-end macro F1': f1_score(status[test], np.where(keep, stage, -100),
                                                          labels=[-100,*range(5)], average='macro', zero_division=0)}
    prec, rec, _ = precision_recall_curve(1-truth, -score)
    line, = ax.step(rec, prec, where='post',
                    label=f"{name} ({binary_rows[name]['Unrelated AP']:.2f} AP)")
    tradeoff.scatter(binary_rows[name]['Orders missed'], binary_rows[name]['Unrelated filtered'],
                     color=line.get_color(), label=name, s=85, alpha=.8)
ax.axhline((1-truth).mean(), ls='--', color='gray', label='Unrelated prevalence')
ax.set(xlabel='Unrelated recall', ylabel='Unrelated precision', title='Detecting unrelated episodes on held-out phones')
ax.legend(fontsize=8)
tradeoff.scatter(0, 0, marker='x', color='black', label='Route everything')
tradeoff.set(xlabel='Order-related episodes missed', ylabel='Unrelated episodes filtered',
             title=f'Test tradeoff at {RECALL_TARGET:.0%} validation order-recall target')
tradeoff.legend(fontsize=8)
display(pd.DataFrame.from_dict(binary_rows, orient='index').round(3))
display(pd.DataFrame.from_dict(status_rows, orient='index').round(3))
plt.show()
ax = pd.DataFrame(stage_f1, index=STATUSES).plot.barh(figsize=(8, 5), xlim=(0, 1),
                                                     title='Status F1 on true order-related episodes')
ax.set_xlabel('F1')
ax.figure.tight_layout()
plt.show()

# Show where errors went, including orders rejected by the relevance gate.
outcomes = [-100, *range(len(STATUSES))]
names = ['unrelated', *STATUSES]
fig, axes = plt.subplots(1, len(systems), figsize=(17, 5), layout='constrained')
for ax, (name, (score, cutoff, stage)) in zip(axes, systems.items()):
    routed = np.where(score >= cutoff, stage, -100)
    counts = confusion_matrix(status[test], routed, labels=outcomes)
    rates = counts / np.maximum(counts.sum(axis=1, keepdims=True), 1)
    ax.imshow(rates, cmap='Blues', vmin=0, vmax=1)
    for i, j in np.ndindex(counts.shape):
        ax.text(j, i, str(counts[i, j]), ha='center', va='center',
                color='white' if rates[i, j] > .55 else 'black', fontsize=8)
    ax.set(title=name, xticks=range(len(names)), yticks=range(len(names)),
           xticklabels=names, yticklabels=names, xlabel='Predicted outcome')
    ax.tick_params(axis='x', labelrotation=45)
axes[0].set_ylabel('Actual outcome')
fig.suptitle('End-to-end test confusion matrices (counts; color normalized by actual class)')
plt.show()


In [ ]:
# Replaced torch.inference_mode() with torch.no_grad() here as well
@torch.no_grad()
def classify_episode(transcript):
    model.eval()
    batch = tokenizer.pad([encode(transcript)], padding='max_length', max_length=LENGTH, return_tensors='pt')
    b, s = model(move(batch))
    score = b.softmax(-1)[0,1].item()
    return dict(is_order=score >= threshold, order_score=score,
                order_status=STATUSES[s.argmax(-1).item()] if score >= threshold else None)

# Synthetic example; pass the same transcript format used in the training files.
print(classify_episode('Customer: Is this medicine available?'))

## Simulating selective LLM use

Fit a word-based relevance filter and a tuned status classifier. Select thresholds on validation phones and estimate request volume and input-token savings on test phones. Token counts use the MuRIL tokenizer as a proxy.

In [ ]:
# Simulate LLM handoff with a word relevance filter and tuned status classifier.
from sklearn.pipeline import make_pipeline

PROMPT_OVERHEAD_TOKENS = 0
with threadpool_limits(limits=1, user_api='blas'):
    word = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    gate = make_pipeline(word, LogisticRegression(C=1, max_iter=500, random_state=SEED))
    gate.fit([rows[i][0] for i in train], order[train])
    vec = FeatureUnion([
        ('word', TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=3, sublinear_tf=True)),
    ])
    x_train = vec.fit_transform([rows[i][0] for i in train])
    x_valid, x_test = [vec.transform([rows[i][0] for i in ix]) for ix in (valid, test)]
    train_orders, valid_orders = order[train] == 1, order[valid] == 1
    best_f1 = -1
    for c, weight in product((0.1, 1, 10), (None, 'balanced')):
        candidate = LogisticRegression(C=c, class_weight=weight, max_iter=500, random_state=SEED)
        candidate.fit(x_train[train_orders], status[train][train_orders])
        value = f1_score(status[valid][valid_orders], candidate.predict(x_valid[valid_orders]),
                         average='macro')
        if value > best_f1:
            best_f1, status_model = value, candidate

scores_by_split = [gate.predict_proba([rows[i][0] for i in ix])[:, 1] for ix in (valid, test)]
stage_probabilities = [status_model.predict_proba(x) for x in (x_valid, x_test)]
reject_below = select_order_cutoff(order[valid], scores_by_split[0], .99)
val_conf = stage_probabilities[0].max(axis=1)
val_pred = status_model.classes_[stage_probabilities[0].argmax(axis=1)]
accept_at = np.inf
for threshold_candidate in np.unique(val_conf):
    keep = (scores_by_split[0] >= reject_below) & (val_conf >= threshold_candidate)
    if keep.sum() >= 20 and np.mean(val_pred[keep] != status[valid][keep]) <= .10:
        accept_at = threshold_candidate
        break

print(f'Auto-unrelated below {reject_below:.4f}. Auto-order at status score >= {accept_at:.4f} among the rest.')
print('Other episodes are deferred in simulation. No API calls.')
summary = []
for name, ids, scores, probs in zip(('Validation', 'Test'), (valid, test),
                                    scores_by_split, stage_probabilities):
    reject = scores < reject_below
    accept = ~reject & (probs.max(axis=1) >= accept_at)
    defer = ~(reject | accept)
    labels = status_model.classes_[probs.argmax(axis=1)]
    summary.append({'Split': name, 'Episodes': len(ids), 'Auto-unrelated': reject.sum(),
                    'Auto-order': accept.sum(), 'LLM episodes': defer.sum(), 'LLM share': defer.mean(),
                    'Orders auto-rejected': (reject & (order[ids] == 1)).sum(),
                    'Auto-order errors': (accept & (labels != status[ids])).sum()})
    if name == 'Test':
        lengths = np.array([len(tokenizer.encode(rows[i][0], add_special_tokens=False, verbose=False))
                            for i in ids]) + PROMPT_OVERHEAD_TOKENS
        print(f'Estimated input token savings: {1 - lengths[defer].sum() / lengths.sum():.1%}')
display(pd.DataFrame(summary).set_index('Split').round(3))
